# LLM Zoomcamp 2026 - Workshop DLT

## Environment Setup
First, we initialize the Logfire integration and the Pydantic AI agent.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

import logfire
from minsearch import Index
import ingest
from agent import faq_agent, SearchDeps

# 1. Load the FAQ data and build the search index
documents = ingest.load_faq_data()
index = Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["course", "id"]
)
index.fit(documents)
deps = SearchDeps(index=index)

# 2. Instrument the agent with Logfire
logfire.configure()
logfire.instrument_pydantic_ai()



## Q1: Number of spans for a single agent run
We run the agent synchronously. A fully successful run with a tool call generally produces ~5 spans (agent span, first llm call, search tool call, second llm call, etc.).

In [2]:
question = 'How do I run Ollama locally?'
print(f"Asking: {question}")

try:
    response = await faq_agent.run(question, deps=deps)
    print("Answer:", response.output)
except Exception as e:
    print("Error:", e)
    





Asking: How do I run Ollama locally?
00:07:28.186 faq_agent run
00:07:28.188   chat gemini-flash-lite-latest


Logfire project URL: https://logfire-us.pydantic.dev/tuanle/llmzoomcamp2026

00:07:28.724   running tool: search
00:07:28.735   chat gemini-flash-lite-latest


Answer: To run Ollama locally, follow these installation steps based on your operating system:

*   **macOS**: Download the `.pkg` file from [ollama.com/download](https://ollama.com/download) and install it.
*   **Windows**: Download the `.msi` file from [ollama.com/download](https://ollama.com/download) and install it.
*   **Linux**: Run the following command in your terminal:
    ```bash
    curl -fsSL https://ollama.com/install.sh | sh
    ```

### How to use it:
1.  **Start a model**: Once installed, open your terminal and run:
    ```bash
    ollama run llama3
    ```
    This will download the model (approx. 4GB) and open a chat interface.

2.  **Verify the server**: You can check if the local server is running by executing:
    ```bash
    curl http://localhost:11434
    ```

3.  **Using it with Python**: You can install the Ollama Python client using `pip install ollama`. Here is a basic example:
    ```python
    import ollama

    response = ollama.chat(
        model='llama3

We can query Logfire directly to check how many spans were produced by the latest run.

In [7]:
import time
print('Waiting for Logfire ingestion...')
time.sleep(5)
import requests

read_token = os.environ.get("LOGFIRE_READ_TOKEN")
headers = {"Authorization": f"Bearer {read_token}"}
url = "https://logfire-api.pydantic.dev/v1/query"

# Get the latest trace_id
trace_query = "SELECT trace_id FROM records ORDER BY start_timestamp DESC LIMIT 1"
resp_trace = requests.get(url, params={"sql": trace_query}, headers=headers).json()
latest_trace_id = resp_trace['columns'][0]['values'][0]

# Count spans for that trace
span_query = f"SELECT COUNT(*) FROM records WHERE trace_id = '{latest_trace_id}'"
resp_spans = requests.get(url, params={"sql": span_query}, headers=headers).json()
span_count = resp_spans['columns'][0]['values'][0]

print(f"Spans produced for trace {latest_trace_id}: {span_count}")


Waiting for Logfire ingestion...
Spans produced for trace 019f807f353a9324f84c1bc8b30ad5e4: 4


## Q2: Load traces into DuckDB with DLT
Using DLT, we fetch the nested JSON records from Logfire and load them into a relational DuckDB database.

In [4]:
import dlt
from dlt.sources.rest_api import RESTAPIConfig, rest_api_resources

@dlt.source
def logfire_source():
    config: RESTAPIConfig = {
        "client": {
            "base_url": "https://logfire-api.pydantic.dev/v1/",
            "auth": {"type": "bearer", "token": read_token},
        },
        "resources": [
            {
                "name": "query",
                "endpoint": {
                    "path": "query",
                    "method": "GET",
                    "params": {"sql": f"SELECT * FROM records WHERE trace_id = '{latest_trace_id}'"}
                }
            }
        ],
    }
    yield from rest_api_resources(config)

pipeline = dlt.pipeline(
    pipeline_name='logfire_pipeline',
    destination='duckdb',
    dataset_name='agent_traces',
)

load_info = pipeline.run(logfire_source())
print("Pipeline loaded successfully.")



2026-07-21 00:07:36,942|[WARNING]|44756|8453397888|dlt|client.py|detect_paginator:365|Fallback `paginator` used: `SinglePagePaginator at 119f79be0`. Please provide paginator manually.


2026-07-21 00:07:37,012|[WARNING]|44756|8453397888|dlt|validate.py|verify_normalized_table:113|In schema `logfire_source`: The following columns in table 'query__values' did not receive any data during this load and therefore could not have their types inferred:
  - model_request_parameters__output_object
  - model_request_parameters__prompted_output_template
  - model_request_parameters__thinking

Unless type hints are provided, these columns will not be materialized in the destination.
One way to provide type hints is to use the 'columns' argument in the '@dlt.resource' decorator.  For example:

@dlt.resource(columns={'model_request_parameters__output_object': {'data_type': 'text'}})



2026-07-21 00:07:37,012|[WARNING]|44756|8453397888|dlt|validate.py|verify_normalized_table:113|In schema `logfire_source`: The following columns in table 'query__values__model_request_parameters__function_tools' did not receive any data during this load and therefore could not have their types inferred:
  - capability_id
  - include_return_schema
  - metadata
  - outer_typed_dict_key
  - return_schema
  - timeout
  - tool_kind
  - unless_native
  - with_native

Unless type hints are provided, these columns will not be materialized in the destination.
One way to provide type hints is to use the 'columns' argument in the '@dlt.resource' decorator.  For example:

@dlt.resource(columns={'capability_id': {'data_type': 'text'}})



Pipeline loaded successfully.


Let's see exactly how many tables DLT automatically created by flattening the nested Logfire structures.

In [8]:
import duckdb

conn = duckdb.connect("logfire_pipeline.duckdb")
sql = """
SELECT COUNT(table_name) 
FROM information_schema.tables 
WHERE table_schema = 'agent_traces'
"""
num_tables = conn.execute(sql).fetchone()[0]
print(f"Number of tables created by dlt: {num_tables}")


Number of tables created by dlt: 26


## Q3: Query traces for Input Token Usage
Finally, we can compute the sum of `gen_ai.usage.input_tokens` across the LLM spans.

In [9]:
# DLT flattens 'attributes.gen_ai.usage.input_tokens' into the column 'gen_ai_usage_input_tokens'
sql_tokens = f"""
SELECT sum(gen_ai_usage_input_tokens) 
FROM agent_traces.query__values
"""
try:
    total_input_tokens = conn.execute(sql_tokens).fetchone()[0]
    print(f"Total input tokens usage captured: {total_input_tokens}")
except Exception as e:
    print("Error querying tokens:", e)
finally:
    conn.close()



Total input tokens usage captured: 1727
